# Import necessary libraries
SAFEP_parse.py contains all the functions and library calls necessary to run the notebook
# Required libraries:
- numpy
- pandas
- matplotlib
- alchemlyb (pip install git+https://github.com/alchemistry/alchemlyb)
- natsort (for sorting file names)
- glob (for unix-like file paths)



In [ ]:
from AFEP_parse import *
plt.rcParams['figure.dpi'] = 150

In [ ]:
import os

In [ ]:
path='../ELIC_Analysis/PCPG31/POCE_1/'
filename='PO*.fepout'
fepoutFiles = glob(path+filename)
temperature = 300
RT = 0.00198720650096 * temperature # ca. 0.59kcal/mol
totalSize = 0
for file in fepoutFiles:
    totalSize += os.path.getsize(file)
print(f"Will process {len(fepoutFiles)} fepout files.\nTotal size:{np.round(totalSize/10**9, 2)}GB")

In [ ]:
def autocorr(X):
    data = X-np.mean(X)
    corr = np.correlate(data, data, 'full')
    corr = np.divide(corr[len(corr)//2:], np.max(corr))
    
    return corr


def decay(x, l, a):
    return a*np.exp(np.divide(x,-1*l))


def getNSamples(data):
    corr = autocorr(data)
    X = np.arange(len(corr))
    popt, pcov = curve_fit(decay, X, upCorr, [1000., 1.])
    decayed = popt[0]*np.log(2)*1.94 #time required for the correlation to be ~95% or less
    size = len(data)
    blockSize = int(decayed)
    nsamples = int(size//blockSize)
    
    return nsamples, blockSize


def getBlockAve(nsamples, data):
    possible = np.arange(nsamples-5, nsamples+5)
    candidates = possible[np.gcd(len(data), possible)==possible]
    idx = abs(candidates-nsamples).argmin()
    optimum = candidates[idx]
    reshapedData = np.reshape(data.values, (optimum, size//optimum))
    reshapedTime = np.reshape(data.index, (optimum, size//optimum))
    blockAve = np.mean(reshapedData, axis=1)
    blockTime = np.mean(reshapedTime, axis=1)
    
    return blockAve, blockTime

In [ ]:
from scipy import signal
import matplotlib.pyplot as plt

fig, ax= plt.subplots(1)
t = df.step[df.up]
sig  = df.dE[df.up]

widths = np.arange(2, blockSize)
cwtmatr = signal.cwt(sig, signal.morlet2, widths)
plt.pcolormesh(cwtmatr.real, cmap='PRGn', vmin=-4, vmax=4)
ax.set(ylabel='width domain', xlabel='time domain')
plt.colorbar()
plt.show()

In [ ]:
upDat = df.dE[df.up]
downDat = df.dE[np.logical_not(df.up)]

In [ ]:
nsamples, blockSize = getNSamples(upDat)
blockAve, blockTime = getBlockAve(nsamples, data)

fig, ax = plt.subplots(1, figsize=(10,5))
ax.plot(data, linewidth=0.5, color='gray')
ax.plot(blockTime, blockAve, linewidth=2, color='blue', label='block averages')
rolling = data.rolling(blockSize, min_periods=0).mean()
ax.plot(rolling, linewidth=2, color='orange', label='rolling average')
ax.legend()

## If we look at the raw dE signal as a function of time for a single window, we see an incomprehensible blur

In [ ]:
fig, (ax1, ax2) = plt.subplots(2,1)
ax1.plot(df.step, df.dE, linewidth=0.25)

y = dcorr[0.01]
x = dcorr.index.get_level_values(0)
ax2.plot(x, y, linewidth = 0.25)

### And we see they have a somewhat skewed distribution

In [ ]:
fig, ax1 = plt.subplots(1,1)
ax1.hist(df.dE, label="raw data", bins=100, density=True, histtype='step')
ax1.hist(y, label="decorrelated data", bins=100, density=True, histtype='step')
plt.legend()

## But if we look at the real portion of the fast fourier transforms...

In [ ]:
plt.plot(rawFFT.real[1:], label="raw FFT", linewidth=0.25)
#plt.yscale("log")
plt.plot(dcorrFFT.real[1:], label="decorrelated FFT", linewidth=0.25)
plt.legend()

In [ ]:
plt.plot(np.cumsum(rawFFT.real))
plt.plot(np.cumsum(dcorrFFT.real))

In [ ]:
y.values

In [ ]:
plt.plot(df.step, np.cumsum(df.dE)/len(df.dE))
plt.plot(x, np.cumsum(y.values)/len(y))

## They're very different in both domain and range

# Small data sets can be read and decorrelated sequentially, if desired
See Shirts and Chodera (2008) for more details

"Statistically optimal analysis of samples from multiple equilibrium states" doi: 10.1063/1.2978177

In [ ]:
maxSize = 10**9 #Don't use the alchemlyb parser if larger than this size. (bytes)
if totalSize < maxSize:
    from alchemlyb.preprocessing import subsampling

    u_nk = namd.extract_u_nk(fepoutFiles, temperature)

    method = 'dE'
    affix = f'decorrelated_{method}'
    #affix = 'unprocessed'

    groups = u_nk.groupby('fep-lambda')
    decorr = pd.DataFrame([])
    for key, group in groups:
        test = subsampling.decorrelate_u_nk(group, method)
        decorr = decorr.append(test)

    u_nk = decorr
else:
    print(f"Warning: The files you are trying to read are quite large. Total size={totalSize}. Try reading and decorrelating (above) or change the maxSize parameter.")

# Carry out MBAR Fitting and Analyses

In [ ]:
bar = BAR()
bar.fit(u_nk)

In [ ]:

test = {}
l = []
x=0
last = 0
for x in np.arange(len(u_nk)):
    ser = u_nk.iloc[x,:]
    idx = ser.last_valid_index()
    if last != idx:
        test[last] = l
        l = []
        
    l.append(ser.dropna()[-1])
    last = idx

In [ ]:
pd.DataFrame.from_dict(test, orient='index')

In [ ]:
start = 0
for key in test:
    end = start+len(test[key])
    X = np.arange(start, end)
    start = end+1
    plt.plot(X, test[key], label=key)

plt.legend()

# Extract key features from the MBAR fitting and get ΔG
Note: alchemlyb operates in units of kT by default. We multiply by RT to conver to units of kcal/mol.

In [ ]:
l, l_mid, f, df, ddf, errors = get_MBAR(bar)
#print("Overall free energy",df.cumsum() * RT) #in unit of kcal/mol
#print("Errors", errors)

#Overall delta G*_site
#print('')
#print('')
print(f'\u0394G = {np.round((df.cumsum()*RT)[-1], 3)}\u00B1{np.round(errors[-1], 3)} kcal/mol')

# Plot the change in free energy based on MBAR estimates

In [ ]:
# Cumulative change in kT
plt.errorbar(l, f, yerr=errors, marker='.')
plt.xlabel('lambda')
plt.ylabel('DeltaG(lambda) (kT)')
plt.title(f'Cumulative dG with accumulated errors {affix}')
plt.savefig(f'{path}dG_cumulative_kT_{affix}.png', dpi=600)
plt.show()

# Cumulative change in kcal/mol
plt.errorbar(l, f * RT, yerr=errors*RT, marker='.')
plt.xlabel('lambda')
plt.ylabel('DeltaG(lambda)(kcal/mol)')
plt.savefig(f'{path}dG_cumulative_kcal_per_mol_{affix}.png', dpi=600)
plt.show()

# Per-window change in kT
plt.errorbar(l_mid, df, yerr=ddf, marker='.')
plt.xlabel('lambda')
plt.ylabel('Delta G per window (kT)')
plt.title(f'Per-Window dG with individual errors {affix}')
plt.savefig(f'{path}dG_{affix}.png', dpi=600)
plt.show()


# Plot the estimated total change in free energy as a function of simulation time; contiguous subsets starting at t=0 ("Forward") and t=end ("Reverse")

In [ ]:
convergence_plot(u_nk, l)
plt.title(f'Convergence {affix}')
plt.savefig(f'{path}convergence_{affix}.png', dpi=600)

# Use an exponential estimator to assess residual discrepancies and check for hysteresis

In [ ]:
l, l_mid, dG_f, dG_b = get_EXP(u_nk)

In [ ]:
plt.vlines(l_mid, np.zeros(len(l_mid)), dG_f + np.array(dG_b), label="fwd - bwd", linewidth=2)

plt.legend()
plt.title(f'Fwd-bwd discrepancies by lambda {affix}')
plt.xlabel('Lambda')
plt.ylabel('Diff. in delta-G')
plt.savefig(f'{path}discrepancies_{affix}.png', dpi=600)

# The above data should follow a roughly normal distribution centered on 0.

In [ ]:
plt.hist(dG_f + np.array(dG_b));
plt.title(f'Distribution of fwd-bwd discrepancies {affix}')
plt.xlabel('Difference in delta-G')
plt.ylabel('Count')
plt.savefig(f'{path}distribution_{affix}.png', dpi=600)